# Evaluation

In [10]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
len(documents) 

72

In [11]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Generating ground truth

In [12]:
# PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
# wget ${PREFIX}/01-agentic-rag/code/rag_helper.py
# wget ${PREFIX}/04-evaluation/code/evaluation_utils.py

In [13]:
import os 
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)
llm_model = "openai/gpt-oss-20b"


In [14]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [15]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [16]:
import json
from evaluation_utils import llm_structured

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model=llm_model
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

In [17]:
filenames_to_generate = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md"
]


usages = []

for filename in filenames_to_generate:
    doc = next(doc for doc in documents if doc["filename"] == filename)
    questions, usage = generate_ground_truth(doc)
    usages.append(usage.input_tokens)
    print(f"Generated {len(questions)} questions for {filename}")

Generated 5 questions for 01-agentic-rag/lessons/01-intro.md
Generated 5 questions for 01-agentic-rag/lessons/02-environment.md
Generated 5 questions for 01-agentic-rag/lessons/03-rag.md


In [18]:
usages[0]

1149

In [19]:
import numpy as np
print(f"the average number of input tokens across these 3 calls? {np.mean(np.array(usages))}")

the average number of input tokens across these 3 calls? 1482.0


In [20]:
# PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
# wget ${PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv

In [21]:
import pandas as pd
ground_truth = pd.read_csv("ground-truth.csv")
ground_truth.head()

,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md
2,What are the main weaknesses of large language...,01-agentic-rag/lessons/01-intro.md
3,What will the course build in the first part o...,01-agentic-rag/lessons/01-intro.md
4,What kind of example app are you building here...,01-agentic-rag/lessons/01-intro.md


In [22]:
# chunking the source documents into smaller pieces for better processing
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [23]:
# for hybrid search
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [24]:
# Creating the index
from minsearch import Index

tindex = Index(text_fields=["content"],keyword_fields=["filename"])
tindex.fit(chunks)


# Creating the index
from minsearch import VectorSearch
from embedder import Embedder

embed = Embedder()

texts = [c['content'] for c in chunks] 
filenames = [c['filename'] for c in chunks]
X = np.array(embed.encode_batch(texts)) 

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [25]:
def text_search(query, num_results=10):
    results = tindex.search(query, num_results=num_results)
    return results

def vector_search(query, num_results=10):
    v = embed.encode(query)
    results = vindex.search(v, num_results=num_results)
    return results

In [26]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [27]:
q = ground_truth.iloc[0]["question"]

In [28]:
text_search(q, num_results=5)[0]

{'start': 3000,
 'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retrieve 

In [29]:
vector_search(q, num_results=5)[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

## Evaluating text search

In [37]:
from tqdm.auto import tqdm

# runs search for a question and returns a list of 0s and 1s
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance


def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

# the fraction of questions where the correct page appears in the results
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

# (Mean Reciprocal Rank) also rewards finding the page near the top
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

# runs a search function over the whole ground truth and returns both metrics
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": round(hit_rate(relevance_total), 2),
        "mrr": round(mrr(relevance_total), 2),
    }

In [40]:
# Evaluating text search

result = evaluate(ground_truth.to_dict(orient="records"), text_search)
result

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.84, 'mrr': 0.61}

In [41]:
# Evaluating vector search

result = evaluate(ground_truth.to_dict(orient="records"), vector_search)
result

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.84, 'mrr': 0.56}

In [45]:
# Evaluating hybrid search
num_lens = [1, 50, 100,  200]

for k in num_lens:
    result = evaluate(ground_truth.to_dict(orient="records"), lambda query: hybrid_search(query, k))
    print(f"k={k}: {result}")

  0%|          | 0/360 [00:00<?, ?it/s]

k=1: {'hit_rate': 0.84, 'mrr': 0.65}


  0%|          | 0/360 [00:00<?, ?it/s]

k=50: {'hit_rate': 0.84, 'mrr': 0.64}


  0%|          | 0/360 [00:00<?, ?it/s]

k=100: {'hit_rate': 0.84, 'mrr': 0.64}


  0%|          | 0/360 [00:00<?, ?it/s]

k=200: {'hit_rate': 0.84, 'mrr': 0.64}
